# 06b-q — eventi ordinati, normalizzazione sparsa e gate passive-default

Un solo run esegue: (A) playground sintetico a risposta nota; (B) matrice appaiata 3×2 rappresentazione degli eventi × normalizzazione; (C) confronto adattivo ungated × passive-default. I ruoli sono train-only, disgiunti per seed/snapshot e bilanciati usando soltanto `U_realized`. Validation, test e fresh test non vengono letti.


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,zipfile
from pathlib import Path
ELM_REPOSITORY='https://github.com/Zagred47/giada.git';DEFAULT_ELM_REF='ea32d9193acdd4fa9b729ca466bed9e5aba34966';ELM_REF=os.environ.get('HAYFLOW_ELM_REF',DEFAULT_ELM_REF)
ROOT=Path('/kaggle/working');WORKSPACE=ROOT/'hayflow_workspace';ELM_REPO=WORKSPACE/'elmneuron';WORKSPACE.mkdir(parents=True,exist_ok=True)
def run(command,cwd=None):print('+',' '.join(map(str,command)),flush=True);subprocess.run(list(map(str,command)),cwd=cwd,check=True)
if not (ELM_REPO/'.git').is_dir():run(['git','clone',ELM_REPOSITORY,ELM_REPO])
run(['git','fetch','origin',ELM_REF],cwd=ELM_REPO);run(['git','checkout','--detach','FETCH_HEAD'],cwd=ELM_REPO)
REVISION=subprocess.check_output(['git','rev-parse','HEAD'],cwd=ELM_REPO,text=True).strip();MODULE_PATH=ELM_REPO/'src/hayflow_model/event_supported_jump_playground.py';assert MODULE_PATH.is_file(),f'Modulo 06b-q assente nel checkout {REVISION}: {MODULE_PATH}';[sys.modules.pop(name,None) for name in tuple(sys.modules) if name=='src' or name.startswith('src.')];sys.path=[str(ELM_REPO)]+[entry for entry in sys.path if entry!=str(ELM_REPO)];importlib.invalidate_caches();print({'revision':REVISION,'module_exists':MODULE_PATH.is_file()})


## 1. Preflight immutabile
Servono dataset composito, 05t, 06b-n, 06b-o e il risultato registrato 06b-p. Gli artefatti sono identificati dal SHA-256 dell'indice anche quando Kaggle li rinomina `archive.zip`.


In [ ]:
from src.hayflow_model.rollout_aware_architecture_canary import discover_indexed_artifact_source,materialize_nested_indexed_artifact_source
from src.hayflow_model.atomic_state_dynamics_playground import EXPECTED_05T_INDEX_SHA256
from src.hayflow_model.effective_membrane_source_playground import EXPECTED_06BN_INDEX_SHA256
from src.hayflow_model.atomic_effective_source_learnability import EXPECTED_06BO_INDEX_SHA256
from src.hayflow_model.event_supported_jump_playground import EXPECTED_06BP_INDEX_SHA256
INPUT_ROOT=Path('/kaggle/input');CACHE=Path('/kaggle/working/.06bq_nested_inputs')
def indexed(label,expected,env):
 override=os.environ.get(env);source=discover_indexed_artifact_source(INPUT_ROOT,expected,override=Path(override) if override else None)
 if source is None:source=materialize_nested_indexed_artifact_source(INPUT_ROOT,expected,CACHE)
 assert source is not None,f'Artefatto {label} esatto non trovato. Aggiungilo agli Input Kaggle oppure imposta {env}.'
 return source
ARTIFACT_05T_SOURCE=indexed('05t',EXPECTED_05T_INDEX_SHA256,'HAYFLOW_05T_ARTIFACT');ARTIFACT_06BN_SOURCE=indexed('06b-n',EXPECTED_06BN_INDEX_SHA256,'HAYFLOW_06BN_ARTIFACT');ARTIFACT_06BO_SOURCE=indexed('06b-o',EXPECTED_06BO_INDEX_SHA256,'HAYFLOW_06BO_ARTIFACT');ARTIFACT_06BP_SOURCE=indexed('06b-p',EXPECTED_06BP_INDEX_SHA256,'HAYFLOW_06BP_ARTIFACT')
def extract_zip_safely(source,destination):
 source,destination=Path(source),Path(destination);marker=destination/'.source_stamp';stamp=f'{source.stat().st_size}:{source.stat().st_mtime_ns}'
 if marker.is_file() and marker.read_text().strip()==stamp:return destination
 if destination.exists():shutil.rmtree(destination)
 destination.mkdir(parents=True);root=destination.resolve()
 with zipfile.ZipFile(source) as archive:
  for member in archive.infolist():
   target=(destination/member.filename).resolve();assert target==root or root in target.parents,member.filename
  archive.extractall(destination)
 marker.write_text(stamp);return destination
topup_override=os.environ.get('HAYFLOW_TOPUP_V3');topup_candidates=([Path(topup_override).expanduser()] if topup_override else [])+list(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))+[p.parent for p in INPUT_ROOT.rglob('composite_dataset_manifest.json')]
TOPUP_SOURCE=next((p.resolve() for p in topup_candidates if p.exists()),None);assert TOPUP_SOURCE is not None,'Top-up BAP v3 non trovato.';TOPUP_ROOT=extract_zip_safely(TOPUP_SOURCE,'/kaggle/working/hayflow06bq_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates=list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'));assert len(manifest_candidates)==1,manifest_candidates;COMPOSITE_MANIFEST=manifest_candidates[0]
base_override=os.environ.get('HAYFLOW_BASE_DATASET');base_candidates=([Path(base_override).expanduser()] if base_override else [])+[p.parent for p in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(p).lower() and 'topup' not in str(p).lower()]+[p for p in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(p).lower()]
BASE_SOURCE=next((p.resolve() for p in base_candidates if p.exists()),None);assert BASE_SOURCE is not None,'Dataset base targeted v1.1 non trovato.'
print({'05t':str(ARTIFACT_05T_SOURCE),'06b-n':str(ARTIFACT_06BN_SOURCE),'06b-o':str(ARTIFACT_06BO_SOURCE),'06b-p':str(ARTIFACT_06BP_SOURCE),'base':str(BASE_SOURCE),'topup':str(TOPUP_SOURCE)})


In [ ]:
from src.hayflow_data import prepare_composite_flowmap_bundle
hash_started={};hash_last={}
def hash_progress(name,done,total):
 now=time.monotonic();hash_started.setdefault(name,now);percent=int(100*done/total)
 if percent>=hash_last.get(name,-10)+10 or done==total:
  elapsed=now-hash_started[name];rate=done/max(elapsed,1e-9);eta=(total-done)/max(rate,1e-9);print(f'[HayFlow 06b-q][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min',flush=True);hash_last[name]=percent
bundle=prepare_composite_flowmap_bundle(COMPOSITE_MANIFEST,base_source=BASE_SOURCE,progress=hash_progress);assert bundle.manifest['valid'] and bundle.transition_count==29880;print({'dataset_valid':True,'transitions':bundle.transition_count,'fingerprint':bundle.fingerprint})


In [ ]:
import yaml
from IPython.display import display
from src.hayflow_model import EventSupportedJumpConfig,EventSupportedJumpPlayground
values=yaml.safe_load((ELM_REPO/'configs/hayflow/hayflow_event_supported_jump_playground.yml').read_text())['event_supported_jump_playground'];config=EventSupportedJumpConfig.from_mapping(values)
OUTPUT_DIR=Path('/kaggle/working/artifacts/hayflow_event_supported_jump_playground');assert not OUTPUT_DIR.exists(),f'Output già presente: {OUTPUT_DIR}. Avvia una sessione nuova per non sovrascrivere artefatti.'
session=EventSupportedJumpPlayground(bundle,OUTPUT_DIR,config,ARTIFACT_05T_SOURCE,ARTIFACT_06BN_SOURCE,ARTIFACT_06BO_SOURCE,ARTIFACT_06BP_SOURCE,code_revision=REVISION)
contract=session.prepare_event_supported_jump_playground();display({'valid':contract['valid'],'role_support':contract['role_support'],'axes':contract['factorial_axes'],'event_scales':contract['event_scales'],'mechanism_factorization':contract['mechanism_factorization_eligibility'],'rate_sidecar':contract['rate_form_state_sidecar']});assert contract['valid'] and not contract['validation_state_accessed'] and not contract['test_state_accessed']


## 2. Playground noto e matrice biologica 3×2
Prima verifichiamo che i sei bracci apprendano un target sintetico noto con sparsità e molteplicità controllate. Poi usiamo minibatch appaiati e bilanciati; i checkpoint sono scelti sulla metà A della calibration e il braccio sulla metà B. I probe di gradiente sono eseguiti dopo il warm-up, non a readout nullo.


In [ ]:
synthetic=session.run_sparse_event_synthetic_preflight();display({'valid':synthetic['valid'],'final_rmse':synthetic['final_rmse'],'checkpoints':synthetic['checkpoints']});assert synthetic['valid']
training=session.train_event_representation_matrix();display({'valid':training['valid'],'selected_arm':training['selected_arm'],'calibration_half_B':training['median_calibration_half_B_rmse_mv']});assert training['valid'] and not training['development_used_during_training']


In [ ]:
evaluation=session.evaluate_event_representation_matrix(training);safety=session.run_passive_default_safety_gate(training);final_report=session.finalize_event_supported_jump_playground(contract,synthetic,training,evaluation,safety)
display({'valid':final_report['valid'],'diagnosis':final_report['diagnosis'],'selected_arm':final_report['selected_calibration_arm'],'selected_gate':final_report['selected_safety_gate'],'one_step_rmse_mv':final_report['median_one_step_rmse_mv'],'one_step_gain':final_report['median_one_step_gain_over_passive_fraction'],'recursive_8ms_rmse_mv':final_report['median_recursive_8ms_rmse_mv'],'recursive_gain':final_report['median_recursive_8ms_gain_over_passive_fraction'],'causal_controls':final_report['causal_control_rmse_increases_mv'],'candidate':final_report['selected_candidate'],'next_step':final_report['next_step']});assert final_report['valid'] and not final_report['validation_state_accessed'] and not final_report['test_state_accessed']


## 3. Download affidabile
La cella usa il metodo Blob/base64 compatibile con Kaggle già validato nel progetto. L'output mostrato resta compatto: non vengono stampati snapshot, tensori o lunghe sequenze di zeri.


In [ ]:
import base64
from IPython.display import Javascript,display
archive_base=Path('/kaggle/working/hayflow_event_supported_jump_playground');archive_path=Path(shutil.make_archive(str(archive_base),'zip',root_dir=OUTPUT_DIR.parent,base_dir=OUTPUT_DIR.name));payload=base64.b64encode(archive_path.read_bytes()).decode('ascii')
javascript=f"""const binary=atob('{payload}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{archive_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),30000);"""
print({'zip':archive_path.name,'size_mib':round(archive_path.stat().st_size/1024**2,2)});display(Javascript(javascript))
